# Hybrid RAG with LangGraph

This is the standard RAG pipeline from `1_standard_rag/5_standard_rag_langgraph.ipynb`, upgraded to **hybrid retrieval**. The graph is identical - `retrieve -> generate` - but the **`retrieve` node now fuses dense (pgvector) and sparse (BM25 via `pg_textsearch`) results with RRF** instead of using vector search alone.

```
question --> [ retrieve: hybrid (pgvector + BM25, fused by RRF) ] --> [ generate ] --> answer
```

Only the retriever changes; generation and everything downstream stay the same. That is the point of modelling RAG as a graph - you can swap one node without touching the others.

Prerequisites:

- Run `1_standard_rag/4_ocr_chunk_store_pgvector.ipynb` - it populates the `rag_documents` collection **and** builds the `bm25_chunks_idx` index.
- `pg_textsearch` installed (see `1_install_pgvector`), and the DekaLLM `OPENAI_API_KEY` set.

## Dependencies

The dense side (`langchain-postgres`, `langchain-huggingface`), the BM25 side (`psycopg`), and generation (`langchain-openai`, `langgraph`) - all in `../requirements.txt`:

```bash
pip install -r requirements.txt
```

## Configuration

Same `.env` as the other notebooks (in-cluster `PG_HOST=pgvector`), plus the DekaLLM key for generation.

```dotenv
PG_HOST=pgvector
PG_PORT=5432
PG_USER=raguser
PG_PASSWORD=change-me-please
PG_DB=ragdb

OPENAI_API_KEY=your-dekallm-api-key
```

In [ ]:
import os
import psycopg
from dotenv import load_dotenv

load_dotenv(".env")

PG_HOST = os.environ.get("PG_HOST", "pgvector")
PG_PORT = os.environ.get("PG_PORT", "5432")
PG_USER = os.environ.get("PG_USER", "raguser")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "change-me-please")
PG_DB = os.environ.get("PG_DB", "ragdb")

CONNECTION = f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
COLLECTION_NAME = "rag_documents"    # collection ingested in notebook 4
INDEX_NAME = "bm25_chunks_idx"       # BM25 index created in notebook 4

# psycopg connection for the BM25 SQL
conn = psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
)
conn.autocommit = True

print("pgvector:", f"{PG_HOST}:{PG_PORT}/{PG_DB}")
print("collection:", COLLECTION_NAME)
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## Connect to the chunks (dense + sparse)

The dense side opens the `rag_documents` collection with the **same embedding model** as ingestion. The sparse side confirms `pg_textsearch` and the `bm25_chunks_idx` index exist and grabs the collection id. Both cover the identical chunks notebook 4 stored.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=CONNECTION,
    use_jsonb=True,
)

with conn.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_extension WHERE extname = 'pg_textsearch';")
    if cur.fetchone() is None:
        raise RuntimeError("pg_textsearch is not installed. See 1_install_pgvector.")
    cur.execute("SELECT 1 FROM pg_indexes WHERE indexname = %s;", (INDEX_NAME,))
    if cur.fetchone() is None:
        raise RuntimeError(f"Index {INDEX_NAME!r} not found. Run notebook 4 first.")
    cur.execute("SELECT uuid FROM langchain_pg_collection WHERE name = %s;", (COLLECTION_NAME,))
    row = cur.fetchone()
    if row is None:
        raise RuntimeError(f"Collection {COLLECTION_NAME!r} not found. Run notebook 4 first.")
    COLLECTION_ID = row[0]

print("dense + sparse ready; collection id:", COLLECTION_ID)

## The hybrid retriever (dense + BM25, fused by RRF)

The same hybrid retriever from `2_hybrid_search_rrf.ipynb`: dense search via pgvector, sparse search via the `bm25_chunks_idx` index, then **Reciprocal Rank Fusion** to merge the two ranked lists using ranks only. `hybrid_search` returns the fused, top-ranked chunk texts.

In [ ]:
def dense_search(query: str, k: int = 5) -> list[str]:
    return [d.page_content for d in vector_store.similarity_search(query, k=k)]


def sparse_search(query: str, k: int = 5) -> list[str]:
    sql = (
        "SELECT document "
        "FROM langchain_pg_embedding "
        "WHERE collection_id = %(cid)s "
        "ORDER BY document <@> to_bm25query(%(q)s, %(idx)s) "   # reads bm25_chunks_idx; best first
        "LIMIT %(k)s;"
    )
    with conn.cursor() as cur:
        cur.execute(sql, {"cid": COLLECTION_ID, "q": query, "idx": INDEX_NAME, "k": k})
        return [r[0] for r in cur.fetchall()]


def rrf_fuse(ranked_lists: list[list[str]], rrf_k: int = 60) -> list[tuple[str, float]]:
    scores: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, key in enumerate(ranked):
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank + 1)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


def hybrid_search(query: str, k_each: int = 5, rrf_k: int = 60, top_n: int = 4) -> list[str]:
    dense = dense_search(query, k_each)
    sparse = sparse_search(query, k_each)
    fused = rrf_fuse([dense, sparse], rrf_k=rrf_k)
    return [text for text, _ in fused[:top_n]]

## The LLM

Generation uses the course model on the DekaLLM OpenAI-compatible endpoint (key from `OPENAI_API_KEY`) - unchanged from the standard-RAG notebook.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b",
    base_url="https://dekallm.cloudeka.ai/",
)

## Graph state and nodes

Compare with the standard-RAG notebook: the state and `generate` node are identical, and **only `retrieve` changed** - it now calls `hybrid_search` instead of `vector_store.similarity_search`. The context is the list of fused chunk texts.

In [ ]:
from typing import TypedDict, List


class RAGState(TypedDict):
    question: str
    context: List[str]
    answer: str


def retrieve(state: RAGState) -> dict:
    chunks = hybrid_search(state["question"], top_n=4)   # <-- hybrid instead of vector-only
    print(f"[retrieve] hybrid selected {len(chunks)} chunks")
    return {"context": chunks}


def generate(state: RAGState) -> dict:
    context_text = "\n\n".join(state["context"])
    messages = [
        ("system",
         "You are a helpful assistant. Answer the question using ONLY the context below. "
         "If the answer is not in the context, say you don't know."),
        ("human", f"Context:\n{context_text}\n\nQuestion: {state['question']}"),
    ]
    answer = llm.invoke(messages).content
    return {"answer": answer}

## Build the graph

Exactly the standard-RAG shape: `START -> retrieve -> generate -> END`. The wiring did not change - only what `retrieve` does inside.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

hybrid_rag = builder.compile()
print(hybrid_rag.get_graph().draw_mermaid())

## Ask a question

`invoke` runs hybrid-retrieve then generate. The answer is grounded in chunks chosen by *both* semantic and keyword relevance.

In [ ]:
result = hybrid_rag.invoke({"question": "What is this document about?"})

print("ANSWER:\n", result["answer"])
print("\nCONTEXT USED:")
for i, chunk in enumerate(result["context"], 1):
    print(f"  {i}. {chunk[:100].replace(chr(10), ' ')}")

### Try your own question

Hybrid retrieval is especially helpful when the question contains a specific keyword or name that a purely semantic search might blur - BM25 pulls that chunk in, and RRF keeps it near the top.

In [ ]:
question = "Summarize the key points in one sentence."
print(hybrid_rag.invoke({"question": question})["answer"])

## Recap

- **Hybrid RAG = standard RAG with a hybrid retriever.** The `retrieve` node fuses pgvector (dense) and BM25 (`pg_textsearch`, sparse) with RRF; `generate` and the graph are unchanged.
- Because RAG is a **graph**, swapping the retrieval step is a one-node change - the rest of the pipeline does not care how the context was found.
- Both retrievers read the **same chunks** from notebook 4 (embeddings + the `bm25_chunks_idx` index), so results are directly fusible.
- Hybrid retrieval gives the best of both worlds: **semantic recall from vectors + exact-term precision from BM25.**

That completes the hybrid RAG pipeline: **ingest once (vectors + BM25) -> hybrid retrieve -> generate.**